## Calibrate Thermal Comfort Response

**Purpose**
Explores thermal comfort response assumptions and their effect on heating demand representations.

**Primary inputs**
- `project/input/stock/buildingstock_example.csv`
- `project/input resources via get_inputs`

**Primary outputs**
- `In-notebook diagnostics and plots`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Thermal Comfort Analysis

**Purpose:** Analyze optimal indoor temperature decisions and marginal heating consumption.

**Project imports:** `project.model`, `project.thermal`.

**Inputs:** Building stock, thermal parameters.

**Outputs:** Temperature-consumption relationship analysis.


In [ ]:
import os


import matplotlib.pyplot as plt
import pandas as pd
from project.model import get_inputs
from project.utils import reindex_mi, format_ax


In [ ]:
resirf_inputs = get_inputs(variables=['buildings', 'energy_prices', 'income'],
                           building_stock=os.path.join('project', 'input', 'stock', 'buildingstock_example.csv'))
buildings = resirf_inputs['buildings']
prices = resirf_inputs['energy_prices'].loc[2018, :]
income = resirf_inputs['income']
income.index.names = ['Income tenant']


### Heating consumption function of indoor temperature


In [ ]:
rslt = {}
temp_int_list = range(10, 25)
for temp_int in temp_int_list:
    heating_consumption = buildings.heating_consumption(temp_int=temp_int)
    heating_consumption *= buildings.surface
    rslt.update({temp_int: heating_consumption})

rslt = pd.DataFrame(rslt)
rslt.T.plot(legend=False)


In [ ]:
temp = buildings.optimal_temperature(prices)
temp


In [ ]:
marginal_heating_consumption = buildings.heating_consumption(marginal=True, climate=None)
# display(marginal_heating_consumption)
marginal_heating_consumption *= buildings.surface
marginal_bill = buildings.energy_bill(prices, marginal_heating_consumption)
marginal_bill


In [ ]:
data = pd.concat((temp, buildings.consumption_actual(prices), marginal_bill, reindex_mi(income, buildings.stock.index)), keys=['opt. temp.', 'conso', 'marginal bill', 'income'], axis=1)
data


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12.8, 9.6))
rslt.T.plot(ax=ax, legend=False, zorder=-1)
data.plot(x='opt. temp.', y='conso', kind='scatter', ax=ax)
format_ax(ax)


$$V(T) = V_{max} . (1 + e^{\alpha (T - T_{min})})$$

FOC:
$$V'(T*) = V_{max} . \alpha . e^{\alpha (T* - T_{min})})$$
